In [0]:
%pip install PyPDF2

In [0]:
dbutils.library.restartPython()

In [0]:
pdf_path = '/Volumes/agentic/agentic_schema/files/01_Data_Files/product_docs/'
files = dbutils.fs.ls(pdf_path)
pdf_files = [f for f in files if f.name.endswith(".pdf")]
for pdf in pdf_files:
    print(" -" + pdf.name)

In [0]:
import PyPDF2
import io


def extract_text_from_pdf(files_path):
    try:
        with open(files_path,"rb") as f:
            pdf_bytes = f.read()

        pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_bytes.encode("latin-1") if isinstance(pdf_bytes, str) else pdf_bytes))

        text_content = ''
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text_content += page.extract_text() + '\n'

        return text_content.strip()
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        return None
    
        

In [0]:
product_data = []

for pdf_file in pdf_files:
    print(f"processing: {pdf_file.name}")

    product_name = pdf_file.name.replace(".pdf","")
    full_path = f"{pdf_path}{pdf_file.name}"
    product_doc = extract_text_from_pdf(full_path)

    if product_doc:
        product_data.append({
            "product_name": product_name,
            "product_doc" : product_doc
            })
        print(f"Successfully completed {len(product_doc)} characters")
    else:
        print("Failed to extract data")
    

In [0]:
df = spark.createDataFrame(product_data,"product_name string,product_doc string")
df.display()

In [0]:
df.write.mode("overwrite").saveAsTable("agentic.agentic_schema.product_docs")